# Vast.ai 2× V100 의료 LLM 서버

5개 모델(Gemma, MedGemma 최종/데이터셋, Qwen, Llama)을 프로젝트의 `models/`에 내려받고 FastAPI로 제공합니다. GPU 0에는 Gemma/Qwen/Llama를 FP16으로, GPU 1에는 MedGemma 공유 base를 FP32로 배치합니다. MedGemma 두 모델은 하나의 base를 공유하며 Adapter만 전환합니다.

Vast.ai 인스턴스에서 이 저장소의 프로젝트 루트를 Jupyter 작업 디렉터리로 연 뒤 셀을 순서대로 실행하세요. 패키지 설치 다음에는 커널을 한 번 재시작해야 합니다.

## 1. V100(sm_70) 지원 CUDA 12.6용 PyTorch와 서빙 패키지 설치

PyTorch 2.11의 CUDA 12.8 wheel은 V100(sm_70)을 포함하지 않습니다. V100 지원이 유지되는 공식 CUDA 12.6 wheel을 고정합니다.

In [ ]:
%pip uninstall -y torch torchvision torchaudio
%pip install --force-reinstall --no-cache-dir torch==2.11.0 --index-url https://download.pytorch.org/whl/cu126
%pip install --upgrade "transformers>=4.51,<5" "peft>=0.20,<1" "accelerate>=1.6,<2" "huggingface_hub>=0.30,<1" "bitsandbytes>=0.45,<1" "protobuf>=5,<7" fastapi uvicorn requests

위 셀이 끝나면 **Kernel → Restart Kernel**을 실행한 뒤 아래부터 계속하세요.

## 2. 프로젝트 경로·비밀값·GPU 확인

토큰을 코드에 직접 적지 않습니다. Hugging Face에서 Gemma, MedGemma, Llama base의 이용 조건에 먼저 동의한 계정의 Read token을 입력하세요.

In [ ]:
import getpass
import os
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
assert (PROJECT_DIR / "scripts" / "vastai_medical_llm_server.py").exists(), (
    "저장소 프로젝트 루트에서 Notebook을 실행하세요: " + str(PROJECT_DIR)
)
MODEL_ROOT = PROJECT_DIR / "models"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face Read token: ")
if not os.environ.get("MODEL_API_KEY"):
    os.environ["MODEL_API_KEY"] = getpass.getpass("서버 Bearer API key(직접 정한 긴 값): ")
if not os.environ["MODEL_API_KEY"]:
    raise ValueError("MODEL_API_KEY는 비워 둘 수 없습니다.")

os.environ["MODEL_ROOT"] = str(MODEL_ROOT)
os.environ["HF_HOME"] = str(MODEL_ROOT / ".hf-cache")
# V100에서는 작은 2~4B 모델을 fp16으로 실행하는 편이 보통 4bit보다 빠릅니다.
os.environ["MODEL_PRECISION"] = "fp16"
print("PROJECT_DIR:", PROJECT_DIR)
print("MODEL_ROOT:", MODEL_ROOT)

In [ ]:
import torch

print("torch:", torch.__version__, "CUDA wheel:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("PyTorch CUDA architectures:", torch.cuda.get_arch_list())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(index, properties.name, f"{properties.total_memory / 1024**3:.1f} GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "V100 GPU 2개가 필요합니다."
assert "sm_70" in torch.cuda.get_arch_list(), "현재 PyTorch wheel은 V100(sm_70)을 지원하지 않습니다. 설치 셀을 다시 실행하고 커널을 재시작하세요."

## 3. 프로젝트의 `models/`에 base와 adapter 다운로드

재실행하면 이미 받은 파일을 재사용합니다. MedGemma base는 두 adapter가 하나를 공유하므로 한 번만 받습니다.

In [ ]:
import requests
from huggingface_hub import snapshot_download

probe = requests.get("https://huggingface.co/api/models/gpt2", timeout=30)
probe.raise_for_status()
print("Hugging Face HTTPS 연결: OK")

ADAPTER_REPOS = {
    "gemma": "ghddls7799/gemma-2-2b-med-ko-qlora",
    "medgemma-final": "gon-0130/medgemma-4b-lora-consultation-main-v2",
    "medgemma-dataset": "gon-0130/medgemma-4b-lora-consultation",
    "qwen": "csj9630/qwen3-4b-medical-qlora",
    "llama": "csj9630/llama32-3b-medical-qlora",
}
BASE_REPOS = {
    "gemma": "google/gemma-2-2b-it",
    "medgemma": "google/medgemma-4b-it",
    "qwen": "Qwen/Qwen3-4B",
    "llama": "meta-llama/Llama-3.2-3B-Instruct",
}

for name, repo_id in ADAPTER_REPOS.items():
    target = MODEL_ROOT / "adapters" / name
    print(f"[adapter] {repo_id} -> {target}")
    snapshot_download(
        repo_id=repo_id, token=os.environ["HF_TOKEN"], local_dir=target, max_workers=8
    )

for name, repo_id in BASE_REPOS.items():
    target = MODEL_ROOT / "base" / name
    print(f"[base] {repo_id} -> {target}")
    snapshot_download(
        repo_id=repo_id, token=os.environ["HF_TOKEN"], local_dir=target, max_workers=8
    )

print("모든 모델 다운로드 완료:", MODEL_ROOT)

## 4. 서버 시작

반드시 worker는 1개만 사용합니다. worker를 늘리면 모델 전체가 다시 로드되어 VRAM이 부족해집니다. 최초 로드에는 몇 분 걸릴 수 있습니다.

In [ ]:
import socket
import subprocess
import sys

with socket.socket() as port_check:
    if port_check.connect_ex(("127.0.0.1", 8000)) == 0:
        raise RuntimeError("8000 포트에 기존 서버가 있습니다. 중복 실행하지 말고 기존 프로세스를 먼저 종료하세요.")

LOG_PATH = PROJECT_DIR / "vastai_llm_server.log"
server_log = LOG_PATH.open("w", buffering=1)
server_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "vastai_medical_llm_server:app",
        "--app-dir", str(PROJECT_DIR / "scripts"),
        "--host", "0.0.0.0", "--port", "8000", "--workers", "1",
    ],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
print("PID:", server_process.pid)
print("로그:", LOG_PATH)

In [ ]:
import time
import requests

HEADERS = {"Authorization": f"Bearer {os.environ['MODEL_API_KEY']}"}
for _ in range(180):
    if server_process.poll() is not None:
        raise RuntimeError(LOG_PATH.read_text()[-5000:])
    try:
        response = requests.get("http://127.0.0.1:8000/health", headers=HEADERS, timeout=3)
        if response.status_code == 200:
            print(response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("15분 안에 서버가 준비되지 않았습니다. 로그를 확인하세요.")

## 5. 동작·속도 확인

첫 호출은 CUDA kernel 준비 때문에 느릴 수 있습니다. 아래 셀을 한 번 실행한 뒤 두 번째 측정값을 기준으로 보세요.

In [ ]:
def call_model(model_id, max_tokens=256):
    started = time.perf_counter()
    response = requests.post(
        "http://127.0.0.1:8000/v1/generate",
        headers=HEADERS,
        json={
            "model": model_id,
            "messages": [{"role": "user", "content": "폐렴의 대표 증상을 짧게 설명해주세요."}],
            "max_output_tokens": max_tokens,
        },
        timeout=300,
    )
    if not response.ok:
        print("HTTP 오류:", response.status_code, response.text)
        if LOG_PATH.exists():
            print("\n--- 서버 로그 마지막 12,000자 ---\n")
            print(LOG_PATH.read_text(errors="replace")[-12000:])
        response.raise_for_status()
    data = response.json()
    elapsed = time.perf_counter() - started
    speed = data["output_tokens"] / elapsed
    print(
        f"\n[{model_id}] GPU {data['gpu']} · {elapsed:.2f}s · {speed:.1f} tok/s"
        f" · output={data['output_tokens']} · finish={data['finish_reason']}\n"
        f"{data['answer']}\n"
    )
    return data

for model_id in ["gemma", "medgemma-final", "medgemma-dataset", "qwen", "llama"]:
    call_model(model_id)

In [ ]:
# 서로 다른 GPU의 두 요청이 실제로 병렬 처리되는지 확인합니다.
from concurrent.futures import ThreadPoolExecutor

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as pool:
    list(pool.map(call_model, ["qwen", "medgemma-final"]))
print("병렬 총 시간:", round(time.perf_counter() - started, 2), "초")

## 6. Vast.ai 외부 연결

Vast.ai 인스턴스의 포트 설정에서 컨테이너 TCP 8000을 공개 포트에 매핑합니다. Backend `.env`에는 `LLM_REMOTE_BASE_URL=http://호스트:공개포트`, `LLM_REMOTE_API_KEY=위에서 입력한 키`, `LLM_REMOTE_ENABLED=true`, `LLM_REMOTE_MAX_CONCURRENCY=5`를 설정합니다. 5개 비교 요청을 서버까지 보내면 서버의 GPU별 잠금이 GPU마다 하나씩 안전하게 처리합니다. 공개 URL이 HTTPS 프록시라면 `https://...` 주소를 그대로 사용하세요.

브라우저에서 `/health`를 바로 열면 Authorization 헤더가 없으므로 401이 정상입니다. Notebook의 인증된 요청이 200인지 확인하세요.